In [1]:
import sys
import os
import MySQLdb

MySQLdb._mysql
# Get the current working directory of the notebook
notebook_dir = os.getcwd()

# Go up one level to the 'root_directory'
root_dir = os.path.dirname(notebook_dir)

# Add the root directory to sys.path
sys.path.append(root_dir)

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "djangowork.settings.dev")

# Setup Django
import django

django.setup()

# Now you can import your model
from store.models import Product

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [3]:
def get_similar_products(product_id: int, top_n: int = 5):
    """
    Returns a list of top N similar Product objects based on text similarity.

    Args:
        product_id (int): The ID of the product to find recommendations for.
        top_n (int): The number of similar products to return.

    Returns:
        QuerySet: A queryset of similar Product instances.
    """

    # Fetch product data from the database
    products = Product.objects.all().values('id', 'title', 'description')

    if not products:
        return []

    # Convert product data to DataFrame
    df = pd.DataFrame(list(products))

    # Combine title and description into one string per product
    df['features'] = df['title'].fillna('') + " " + df['description'].fillna('')

    # Convert text to TF-IDF vectors
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(df['features'])

    # Check if the given product_id exists
    if product_id not in df['id'].values:
        return []

    # Find index of the product
    idx = df[df['id'] == product_id].index[0]

    # Calculate cosine similarity of this product with all products
    cosine_sim = linear_kernel(tfidf_matrix[idx], tfidf_matrix)

    # Get the indices of top N most similar products (excluding the product itself)
    sim_scores = list(enumerate(cosine_sim[0]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    similar_indices = [i[0] for i in sim_scores[1:top_n + 1]]

    # Get product IDs for those similar indices
    similar_ids = df.iloc[similar_indices]['id'].tolist()

    return Product.objects.filter(id__in=similar_ids)